In [ ]:
"""
upi_simulator_datapump.py
A high-throughput Python simulator and datapump for synthetic UPI/eFRM transaction events.
Publishes realistic synthetic transactions to a Kafka topic at a controlled TPS.

Usage:
    python upi_simulator_datapump.py --time 60 --threads 8 --tps 1000 --topic upi-events --bootstrap-servers localhost:9092
"""

import argparse
import json
import random
import time
import threading
import uuid
from datetime import datetime
from kafka import KafkaProducer

# ==========================================
# Synthetic Master Data (Subset for Speed)
# ==========================================
APPS = ["com.phonepe.app", "net.one97.paytm", "in.org.npci.upiapp", "com.google.android.apps.nbu.paisa.user", "in.amazon.mShop.android.shopping"]
BANKS = ["HDFC", "ICIC", "SBIN", "UTIB", "KKBK", "YESB", "PUNB", "CNRB"]
HANDLES = ["@ybl", "@axl", "@okhdfcbank", "@oksbi", "@okicici", "@paytm", "@ibl", "@upi"]
STATES = ["MAHARASHTRA", "KARNATAKA", "TAMIL_NADU", "DELHI", "GUJARAT", "WEST_BENGAL", "RAJASTHAN"]
CITIES = ["MUMBAI", "BENGALURU", "CHENNAI", "NEW_DELHI", "AHMEDABAD", "KOLKATA", "JAIPUR"]
TXN_TYPES = ["P2P", "P2M", "P2A", "COLLECT", "MANDATE"]
CHANNELS = ["UPI_APP", "QR", "INTENT", "COLLECT", "IN_APP"]
AUTH_METHODS = ["UPI_PIN", "BIOMETRIC", "DEVICE_AUTH", "OTP"]

# ==========================================
# Event Generator
# ==========================================
def generate_upi_event():
    """Generates a synthetic UPI/eFRM transaction event."""
    txn_id = f"TXN{uuid.uuid4().hex[:14].upper()}"
    timestamp = datetime.utcnow().isoformat() + "Z"
    
    # Log-normal distribution for realistic amount skew (mostly small, some large)
    amount = round(random.lognormvariate(4.5, 1.2), 2)
    amount = min(max(amount, 1.0), 200000.0)
    
    payer_id = f"P{random.randint(10000000, 99999999)}"
    payee_id = f"B{random.randint(10000000, 99999999)}"
    
    payer_vpa = f"{payer_id.lower()}{random.choice(HANDLES)}"
    payee_vpa = f"{payee_id.lower()}{random.choice(HANDLES)}"
    
    # Simulate risk scoring (correlated with fraud)
    risk_score = round(random.uniform(0, 100), 2)
    if random.random() < 0.02:  # 2% chance of high risk spike
        risk_score = round(random.uniform(75, 100), 2)
        
    is_fraud = 1 if risk_score > 85 and random.random() > 0.3 else 0
    
    event = {
        "txn_id": txn_id,
        "timestamp": timestamp,
        "payer_vpa": payer_vpa,
        "payee_vpa": payee_vpa,
        "payer_bank": random.choice(BANKS),
        "payee_bank": random.choice(BANKS),
        "amount": amount,
        "currency": "INR",
        "transaction_type": random.choice(TXN_TYPES),
        "payment_channel": random.choice(CHANNELS),
        "app": random.choice(APPS),
        "device_id": uuid.uuid4().hex,
        "ip_address": f"{random.randint(10,250)}.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}",
        "state": random.choice(STATES),
        "city": random.choice(CITIES),
        "auth_method": random.choice(AUTH_METHODS),
        "risk_score": risk_score,
        "risk_level": "CRITICAL" if risk_score >= 75 else "HIGH" if risk_score >= 50 else "MEDIUM" if risk_score >= 25 else "LOW",
        "is_fraud": is_fraud,
        "processing_status": "SUCCESS" if random.random() > 0.05 else "FAILURE",
        "response_code": "00" if random.random() > 0.05 else random.choice(["U30", "FR", "91"]),
        "merchant_id": f"MER{random.randint(100000, 999999)}" if random.random() > 0.4 else None
    }
    return event

# ==========================================
# Worker Thread
# ==========================================
def worker_thread(thread_id, producer, topic, stop_event, start_time, duration, tps, num_threads, stats):
    """Worker function to generate and send events."""
    # Calculate sleep time per thread to maintain global TPS without lock contention
    sleep_duration = num_threads / tps if tps > 0 else 0
    count = 0
    
    while not stop_event.is_set() and (time.monotonic() - start_time) < duration:
        loop_start = time.monotonic()
        
        # Generate event
        event = generate_upi_event()
        event['simulator_thread'] = thread_id
        
        # Send to Kafka (Asynchronous)
        try:
            producer.send(topic, value=event)
            count += 1
        except Exception as e:
            print(f"[Thread {thread_id}] Error sending to Kafka: {e}")
            
        # Calculate remaining sleep time to maintain exact TPS
        elapsed = time.monotonic() - loop_start
        sleep_time = sleep_duration - elapsed
        if sleep_time > 0:
            time.sleep(sleep_time)
            
    stats[thread_id] = count

# ==========================================
# Main Execution
# ==========================================
def main():
    parser = argparse.ArgumentParser(description="UPI/eFRM Synthetic Data Simulator & Kafka Datapump")
    parser.add_argument("-t", "--time", type=int, default=60, help="Duration to run the simulator in seconds (default: 60)")
    parser.add_argument("-th", "--threads", type=int, default=4, help="Number of concurrent threads (default: 4)")
    parser.add_argument("-r", "--tps", type=int, default=100, help="Target transactions per second (total across all threads, default: 100)")
    parser.add_argument("--topic", type=str, default="upi-efrm-events", help="Kafka topic to publish to (default: upi-efrm-events)")
    parser.add_argument("--bootstrap-servers", type=str, default="localhost:9092", help="Kafka bootstrap servers (default: localhost:9092)")
    
    args = parser.parse_args()
    
    print(f"Starting UPI Simulator...")
    print(f"Target TPS: {args.tps} | Threads: {args.threads} | Duration: {args.time}s")
    print(f"Kafka Topic: {args.topic} | Servers: {args.bootstrap-servers}")
    print("-" * 50)
    
    # Initialize Kafka Producer
    producer = KafkaProducer(
        bootstrap_servers=args.bootstrap_servers.split(','),
        value_serializer=lambda m: json.dumps(m).encode('utf-8'),
        acks='1',
        linger_ms=5,          # Batch slightly for better throughput
        batch_size=16384,
        compression_type='snappy'
    )
    
    stop_event = threading.Event()
    stats = {}
    threads = []
    start_time = time.monotonic()
    
    # Start worker threads
    for i in range(args.threads):
        t = threading.Thread(
            target=worker_thread, 
            args=(i, producer, args.topic, stop_event, start_time, args.time, args.tps, args.threads, stats)
        )
        t.start()
        threads.append(t)
        
    # Monitor progress
    try:
        while time.monotonic() - start_time < args.time:
            time.sleep(1)
            elapsed = time.monotonic() - start_time
            total_sent = sum(stats.values())
            current_tps = total_sent / elapsed if elapsed > 0 else 0
            print(f"[{elapsed:.1f}s] Total Sent: {total_sent} | Current Avg TPS: {current_tps:.1f}")
    except KeyboardInterrupt:
        print("\nInterrupted by user. Stopping...")
        stop_event.set()
        
    # Wait for all threads to finish
    for t